# MNIST FPGA Live Classification Implementation

This notebook documents the development of the first working **live local MNIST classifier on the ZedBoard**.

It begins **after** two pieces had already been completed and verified:

1. the standalone OV7670 camera/VGA project was working; and
2. the MNIST neural-network files had already been generated using the ZyNet model-development workflow.

The purpose of this notebook is to document how the neural network was integrated into the camera project, how a 28 × 28 camera-derived image was captured and frozen correctly in FPGA BRAM, how the frozen image was presented to ZyNet, and how the resulting digit classification was displayed on the ZedBoard LEDs.

This notebook intentionally does **not** cover formal latency, throughput, or controlled-dataset benchmarking. Those experiments were performed in later branches.


## 1. Implementation context

The starting point for this branch was a self-contained camera design that already provided:

- OV7670 configuration and capture;
- VGA display;
- board clock generation;
- push-button input;
- camera-frame buffering; and
- verified live video output.

The trained MNIST model had also already been converted into FPGA-compatible ZyNet files.

The integration goal was therefore:

```text
OV7670 camera
    ↓
camera capture
    ↓
28 × 28 sampled image
    ↓
stored BRAM
    ↓
ZyNet neural network
    ↓
predicted digit
    ↓
ZedBoard LEDs
```

The deployed network architecture was:

```text
784 → 30 → 20 → 10
```

with sigmoid activation and a hardmax/maximum-selection output stage.


## 2. Source brought into the camera project

The camera project was extended with the generated/supporting ZyNet RTL:

```text
zynet.v
include.v
Layer_1.v
Layer_2.v
Layer_3.v
neuron.v
Weight_Memory.v
Sig_ROM.v
maxFinder.v
relu.v
axi_lite_wrapper.v
param_loader.v
weights.mem
biases.mem
```

The existing camera-side source remained responsible for OV7670 configuration, capture, VGA display, clocking, and board I/O.

The integrated top-level design is the camera `ov7670_top` source.

Two Block Memory Generator configurations were kept:

```text
blk_mem_gen_0.xci
blk_mem_gen_stored.xci
```

The second provides storage for the 28 × 28 image that is frozen and later read by both VGA display logic and the neural-network input scanner.


## 3. Behavioral simulation remained part of the ZyNet workflow

Although the Vivado project structure was changed to support the camera design, the ZyNet behavioral-simulation workflow was still used to verify that the generated network operated correctly.

The `genTestData.py` reads `mnist.pkl.gz`, converts MNIST test images into the configured 8-bit representation, and writes simulation input files into the Vivado/XSim behavioral-simulation directory.

The historical workflow was:

1. open the integrated Vivado project;
2. run behavioral simulation;
3. generate the MNIST test vectors with `genTestData.py`;
4. rerun behavioral simulation; and
5. verify the neural-network output before moving to hardware.

The simulation test-data generator is kept as **verification support**, even though it is not part of the live camera inference path.


## 4. Capture a 28 × 28 image from the live camera

The live camera frame was larger than the 28 × 28 MNIST input expected by the neural network.

The working design sampled a **112 × 112 region of interest** from the camera frame:

```text
columns: 24 through 135
rows:     4 through 115
```

Only every fourth row and every fourth column were kept. This produces:

```text
112 / 4 = 28 columns
112 / 4 = 28 rows

28 × 28 = 784 pixels
```

The stored image therefore has exactly the number of values required by the 784-input MNIST network.


In [ ]:
roi_width = 136 - 24
roi_height = 116 - 4
sample_stride = 4

stored_width = roi_width // sample_stride
stored_height = roi_height // sample_stride
stored_pixels = stored_width * stored_height

print("ROI:", roi_width, "x", roi_height)
print("Stored:", stored_width, "x", stored_height)
print("Pixels:", stored_pixels)


### Pixel representation

The camera capture path provides a 12-bit pixel value. The stored BRAM input uses the upper four bits duplicated to form an 8-bit value:

```vhdl
stored_dina <= capture_data(11 downto 8) &
               capture_data(11 downto 8);
```

This provided a compact intensity representation for the stored 28 × 28 image used by the local classifier.


## 5. The first major problem: freezing at the wrong time

The early versions could stop updating the stored image as soon as the freeze control changed state. That meant the 28 × 28 buffer could contain portions of different camera frames or an incomplete sampled image.

The development photos document this progression:

```text
02_early_frozen_image_strip_artifact.jpg
03_early_frozen_image_capture_artifact.jpg
08_frame_timing_issue.jpg
09_resolving_frozen_frame_retrieval.jpg
```

These are development/debugging images rather than classification results.


### Final freeze behavior

The working implementation changed the freeze operation into a **pending request**.

A button press requested a freeze, but the design continued writing sampled pixels until the current 28 × 28 ROI reached its final sampled position.

The working source waits for:

```vhdl
c_int = 132
r_int = 112
```

before asserting the frozen state.

Conceptually:

```text
button pressed
    ↓
freeze requested
    ↓
continue capturing current ROI
    ↓
reach final sampled ROI pixel
    ↓
freeze stored BRAM
```

This ensured that the stored 784-pixel buffer represented one complete sampled camera frame rather than a partially updated frame.


## 6. Crossing from the camera clock domain to the NN/VGA clock domain

Image capture is driven by the buffered camera pixel clock, while stored-image display and neural-network scanning operate in the 25 MHz domain.

The frozen-state signal is synchronized into the `clk25` domain:

```vhdl
freeze_sync_clk25 <=
    freeze_sync_clk25(1 downto 0) & freeze_stored_pclk;

freeze_stored_clk25 <= freeze_sync_clk25(2);
```

This prevents the `clk25` control logic from directly consuming an asynchronous camera-domain control signal.


## 7. Displaying the frozen image and reading it for inference

The stored 28 × 28 image served two purposes:

1. visually confirming what the classifier would receive; and
2. supplying the 784 pixel values to ZyNet.

A 28 × 28 VGA region was defined for the stored image:

```text
horizontal: 300–327
vertical:   200–227
```

The stored BRAM read port was shared between the VGA thumbnail and the neural-network scanner.

In the final working control scheme:

- `sw_view_stored = 1` gives the stored-image display logic access to the BRAM read address;
- `sw_view_stored = 0` gives the neural-network scanner access to the BRAM read address.

Conceptually:

```vhdl
addr_to_stored_bram <= stored_addr_read
                       when sw_view_stored = '1'
                       else nn_scan_addr;
```

This resolved the conflict between displaying the stored image and streaming that same BRAM through the neural network.


## 8. Trigger one neural-network scan of the frozen image

Once a new complete frame had been frozen, the working design marked a neural-network run as pending.

The classifier was not started while VGA still owned the stored-BRAM read port. Instead, the scan began after the frozen image had been obtained and BRAM read access was returned to the NN.

The workflow became:

```text
capture/freeze
      ↓
optional visual inspection of frozen thumbnail
      ↓
give BRAM read port to NN
      ↓
scan one 784-pixel image
```


## 9. Feed the stored BRAM into ZyNet

The NN input scanner uses a small state machine:

```text
NN_IDLE
   ↓
NN_ADDR
   ↓
NN_SEND
   ↺ until all 784 pixels are sent
```

`NN_ADDR` provides a BRAM-read latency cycle.

`NN_SEND` presents the returned BRAM value to ZyNet and asserts input-valid:

```vhdl
nn_data_in  <= stored_doutb;
nn_valid_in <= '1';
```

The scan address advances from 0 through 783, corresponding to the complete 28 × 28 stored image.


## 10. Capture and display the classification result

When `nn_out_valid` is asserted, the predicted digit is latched:

```vhdl
nn_result_latch <= nn_out(3 downto 0);
nn_out_seen     <= '1';
```

The live implementation maps LEDs as:

```text
LED0       camera configuration complete
LED1       frozen-image state
LED2       NN scan complete
LED3       NN result observed
LED7..4    predicted digit
```

This made it possible to verify the entire path rather than only the final class.


## 11. Development photo sequence

The data branch contains:

```text
01_live_camera_feed_setup.jpg
02_early_frozen_image_strip_artifact.jpg
03_early_frozen_image_capture_artifact.jpg
04_live_roi_digit_capture.jpg
05_improved_frozen_digit_roi.jpg
06_zedboard_ov7670_hardware_setup.jpg
07_live_classification_system_setup.jpg
08_frame_timing_issue.jpg
09_resolving_frozen_frame_retrieval.jpg
10_working_live_digit_roi.jpg
11_final_live_setup.jpg
12_preclassification_1.jpg
13_frozen_1_thumbnail.jpg
14_classified_1.jpg
15_classified_5.jpg
```

These document the progression from a working live feed, through frozen-frame timing/retrieval problems, to stable stored-image capture and successful local classification.


### Example: preclassification and frozen-image verification

![Physical digit before classification](12_preclassification_1.jpg)

![Frozen 28x28-derived image](13_frozen_1_thumbnail.jpg)

The frozen/stored representation shows the actual camera-derived input used by the local classifier.


### Successful local classifications

![Successful classification of digit 1](14_classified_1.jpg)

![Successful classification of digit 5](15_classified_5.jpg)

These photographs document successful **functional examples**. They are not part of the later controlled MNIST accuracy benchmark.


## 12. Live demonstration videos

Two videos are kept:

```text
classifying_1_live.mp4
classifying_5_live.mp4
```

They document successful live local FPGA classification of physical digits 1 and 5.

- [Live classification of digit 1](classifying_1_live.mp4)
- [Live classification of digit 5](classifying_5_live.mp4)


## 13. Implementation artifacts kept with the data branch

The associated data branch preserves:

- generated `.mif` files;
- behavioral-simulation test vectors;
- final bitstream;
- implemented schematic;
- utilization reports;
- timing reports;
- power reports;
- route-status report;
- development photographs; and
- live-classification videos.

These provide an audit trail from source integration through implementation and functional hardware validation.


## 14. Final live implementation result

The completed implementation demonstrated:

```text
physical digit
    ↓
OV7670 image capture
    ↓
112 × 112 ROI
    ↓
every fourth row/column sampled
    ↓
28 × 28 stored image
    ↓
complete-frame freeze
    ↓
stored-BRAM scan
    ↓
ZyNet inference
    ↓
LED classification result
```

The principal implementation challenge was obtaining a **stable, complete, camera-derived 28 × 28 image** and safely sharing that stored image between visual verification and the NN input path.

Once the freeze timing and stored-BRAM read ownership were corrected, the design successfully classified physical MNIST-style digits locally on the FPGA.


## 15. Scope boundary

This notebook documents **functional live classification only**.

It does not report:

- formal inference latency;
- throughput;
- 1,000-image hardware accuracy;
- repeated frozen-image timing;
- cloud-assisted inference; or
- cloud communication performance.

Those measurements were performed in later experimental branches using controlled stored MNIST inputs.
